# 🎨 IRO Studio — AI Photo Enhancer Training Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cnttdlpt/photoedit/blob/main/ai_training/iro_ai_photo_enhancer.ipynb)

Chào mừng bạn đến với Notebook huấn luyện **AI Chuyên Chỉnh Ảnh Thông Minh** cho **IRO Studio**!

### 🌟 Mục tiêu của mô hình:
1. **Phân tích hình ảnh**: Nhận diện mức độ phơi sáng, tương phản, nhiệt độ màu K, vùng sáng/tối và sắc thái của bức ảnh.
2. **Dự đoán 10 thông số thanh trượt tối ưu**: Độ sáng, Tương phản, Highlights, Shadows, Whites, Blacks, Nhiệt độ K, Sắc độ Tint, Bão hòa màu và Trong khối (Clarity).
3. **Gợi ý phim Fujifilm chuẩn xác**: Tự động chọn 1 trong 8 giả lập phim trứ danh (Classic Chrome, Velvia, Astia, Classic Neg, Provia, Eterna, Acros, Nostalgic Neg).
4. **Xuất mô hình chuẩn ONNX siêu nhẹ (< 2MB)** để chạy trực tiếp 100% trên trình duyệt Web (ONNX Runtime Web) với tốc độ **20ms**!

## 🛠️ Bước 1: Cài đặt thư viện và chuẩn bị môi trường GPU

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q torch torchvision onnx onnxruntime pillow numpy matplotlib

import os
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Đang sử dụng thiết bị tính toán: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU Model: {torch.cuda.get_device_name(0)}")

## 🧠 Bước 2: Định nghĩa Kiến Trúc Mạng Nơ-ron (IRO-EnhanceNet)

Sử dụng backbone **MobileNetV3-Small** siêu nhẹ, tối ưu hóa cho di động và web.

In [ ]:
FILM_CLASSES = [
    'Classic Chrome',
    'Velvia',
    'Astia',
    'Classic Neg.',
    'PROVIA',
    'ETERNA',
    'Acros',
    'Nostalgic Neg.'
]

PARAM_RANGES = [
    ('exposure', -2.0, 2.0),
    ('contrast', -50.0, 50.0),
    ('highlights', -100.0, 100.0),
    ('shadows', -100.0, 100.0),
    ('whites', -100.0, 100.0),
    ('blacks', -100.0, 100.0),
    ('temp', -50.0, 50.0),
    ('tint', -50.0, 50.0),
    ('saturation', -50.0, 50.0),
    ('clarity', 0.0, 50.0),
]

class IROEnhanceNet(nn.Module):
    def __init__(self, num_params=10, num_films=8, pretrained=True):
        super().__init__()
        weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        base = models.mobilenet_v3_small(weights=weights)
        self.features = base.features
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        in_dim = 576
        
        self.shared_fc = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(inplace=True),
            nn.Dropout(p=0.2)
        )
        
        # Head 1: Dự đoán 10 thanh trượt trong khoảng [-1, 1]
        self.param_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.Hardswish(inplace=True),
            nn.Linear(128, num_params),
            nn.Tanh()
        )
        
        # Head 2: Phân loại giả lập phim Fujifilm
        self.film_head = nn.Sequential(
            nn.Linear(256, 64),
            nn.Hardswish(inplace=True),
            nn.Linear(64, num_films)
        )

    def forward(self, x):
        feat = self.features(x)
        feat = self.avgpool(feat)
        feat = torch.flatten(feat, 1)
        shared = self.shared_fc(feat)
        params = self.param_head(shared)
        film_logits = self.film_head(shared)
        return params, film_logits

model = IROEnhanceNet(pretrained=True).to(device)
print("✅ Mô hình IRO-EnhanceNet đã sẵn sàng!")
total_params = sum(p.numel() for p in model.parameters())
print(f"📊 Tổng số tham số mạng: {total_params:,} (Siêu nhẹ ~ {total_params*4 / (1024*1024):.2f} MB)")

## 📦 Bước 3: Bộ Dữ Liệu Tự Sinh (Self-Synthesizing Dataset)

Tự động tạo ra hàng nghìn mẫu ảnh có điều kiện ánh sáng, màu sắc ngẫu nhiên để mô hình học cách phân tích và điều chỉnh về mức cân bằng hoàn hảo.

In [ ]:
class SyntheticPhotoDataset(Dataset):
    def __init__(self, size=4000, img_size=224):
        self.size = size
        self.img_size = img_size
        self.transform = T.Compose([
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        np_img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        grad_dir = np.random.choice(['horiz', 'vert', 'radial'])
        if grad_dir == 'vert':
            col1 = np.random.randint(20, 240, size=3)
            col2 = np.random.randint(20, 240, size=3)
            for c in range(3):
                np_img[:, :, c] = np.linspace(col1[c], col2[c], self.img_size, dtype=np.uint8)
        elif grad_dir == 'horiz':
            col1 = np.random.randint(20, 240, size=3)
            col2 = np.random.randint(20, 240, size=3)
            for c in range(3):
                np_img[:, :, c] = np.linspace(col1[c], col2[c], self.img_size, dtype=np.uint8)[None, :]
        else:
            np_img[:] = np.random.randint(40, 220, size=3, dtype=np.uint8)

        for _ in range(np.random.randint(2, 6)):
            x = np.random.randint(0, self.img_size)
            y = np.random.randint(0, self.img_size)
            r = np.random.randint(20, 80)
            col = np.random.randint(0, 255, size=3)
            yy, xx = np.ogrid[:self.img_size, :self.img_size]
            mask = (xx - x)**2 + (yy - y)**2 <= r**2
            np_img[mask] = col

        pil_img = Image.fromarray(np_img)
        
        r_mean = np.mean(np_img[:, :, 0])
        g_mean = np.mean(np_img[:, :, 1])
        b_mean = np.mean(np_img[:, :, 2])
        lum = 0.299 * r_mean + 0.587 * g_mean + 0.114 * b_mean
        
        target_exposure = np.clip((128.0 - lum) / 64.0, -2.0, 2.0)
        target_contrast = np.clip((lum - 128.0) * 0.3, -30.0, 30.0)
        target_highlights = np.clip((180.0 - np.percentile(np_img, 95)) * 0.5, -80.0, 80.0)
        target_shadows = np.clip((np.percentile(np_img, 10) - 40.0) * 0.5, -80.0, 80.0)
        target_whites = np.clip((240.0 - np.max(np_img)) * 0.4, -60.0, 60.0)
        target_blacks = np.clip((np.min(np_img) - 15.0) * 0.4, -60.0, 60.0)
        target_temp = np.clip((b_mean - r_mean) * 0.6, -45.0, 45.0)
        target_tint = np.clip(((r_mean + b_mean)/2.0 - g_mean) * 0.5, -45.0, 45.0)
        color_std = np.std(np_img)
        target_saturation = np.clip(15.0 - color_std * 0.2, -30.0, 40.0)
        target_clarity = np.clip(20.0, 5.0, 40.0)
        
        if target_temp > 10:
            target_film = 0 // Classic Chrome
        elif b_mean > r_mean + 20:
            target_film = 3 // Classic Neg.
        elif color_std > 60:
            target_film = 1 // Velvia
        elif lum < 80:
            target_film = 5 // ETERNA
        else:
            target_film = 4 // PROVIA

        raw_params = np.array([
            target_exposure, target_contrast, target_highlights, target_shadows,
            target_whites, target_blacks, target_temp, target_tint,
            target_saturation, target_clarity
        ], dtype=np.float32)

        norm_params = np.zeros(10, dtype=np.float32)
        for i, (_, p_min, p_max) in enumerate(PARAM_RANGES):
            norm_params[i] = 2.0 * (raw_params[i] - p_min) / (p_max - p_min) - 1.0

        return self.transform(pil_img), torch.tensor(norm_params, dtype=torch.float32), torch.tensor(target_film, dtype=torch.long)

dataset = SyntheticPhotoDataset(size=5000)
train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, pin_memory=True if torch.cuda.is_available() else False)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
print(f"✅ Đã chuẩn bị {train_size} mẫu huấn luyện và {val_size} mẫu kiểm định.")

## 🚀 Bước 4: Vòng Lặp Huấn Luyện (Training Loop)

In [ ]:
epochs = 15
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

criterion_params = nn.SmoothL1Loss()
criterion_film = nn.CrossEntropyLoss()

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_loss = float('inf')

print("🏁 Bắt đầu huấn luyện...")
for epoch in range(1, epochs + 1):
    t0 = time.time()
    model.train()
    train_loss = 0.0
    for imgs, params, films in train_loader:
        imgs, params, films = imgs.to(device), params.to(device), films.to(device)
        optimizer.zero_grad()
        pred_params, pred_films = model(imgs)
        loss = criterion_params(pred_params, params) + 0.3 * criterion_film(pred_films, films)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(imgs)
    train_loss /= len(train_ds)
    scheduler.step()

    model.eval()
    val_loss = 0.0
    val_acc = 0.0
    with torch.no_grad():
        for imgs, params, films in val_loader:
            imgs, params, films = imgs.to(device), params.to(device), films.to(device)
            pred_params, pred_films = model(imgs)
            loss = criterion_params(pred_params, params) + 0.3 * criterion_film(pred_films, films)
            val_loss += loss.item() * len(imgs)
            val_acc += (pred_films.argmax(dim=1) == films).sum().item()
    val_loss /= len(val_ds)
    val_acc /= len(val_ds)
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Epoch [{epoch:02d}/{epochs:02d}] ({elapsed:.1f}s) - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Film Acc: {val_acc*100:.1f}%")
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')

print(f"\n🎉 Huấn luyện thành công! Best Loss: {best_loss:.4f}")

## 📊 Bước 5: Kiểm Tra Thử Nghiệm Dự Đoán (Inference Check)

In [ ]:
model.eval()
sample_img, target_p, target_f = val_ds[0]

with torch.no_grad():
    pred_p, pred_f = model(sample_img.unsqueeze(0).to(device))
    pred_p = pred_p.cpu().numpy()[0]
    pred_film_idx = pred_f.cpu().argmax().item()

# Giải mã thông số về dải thực tế
predicted_settings = {}
for i, (name, p_min, p_max) in enumerate(PARAM_RANGES):
    val = (pred_p[i] + 1.0) / 2.0 * (p_max - p_min) + p_min
    predicted_settings[name] = round(float(val), 2)

print("✨ KẾT QUẢ AI DỰ ĐOÁN TỰ ĐỘNG CHO ẢNH:")
print(f"🎞️ Phim đề xuất: {FILM_CLASSES[pred_film_idx]}")
for k, v in predicted_settings.items():
    print(f"  • {k.ljust(12)}: {v}")

## 📦 Bước 6: Xuất Mô Hình Chuẩn ONNX (Dành cho Web)

Tạo tệp `iro_ai_enhancer.onnx` để nhúng vào IRO Studio.

In [ ]:
model.eval()
model.cpu()
dummy_input = torch.randn(1, 3, 224, 224, dtype=torch.float32)
onnx_path = 'iro_ai_enhancer.onnx'

print("⏳ Đang xuất file ONNX...")
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['sliders', 'film_logits'],
    dynamic_axes={'input': {0: 'batch_size'}, 'sliders': {0: 'batch_size'}, 'film_logits': {0: 'batch_size'}}
)

size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"✅ Đã tạo thành công {onnx_path} ({size_mb:.2f} MB)!")

# Kiểm tra lại bằng ONNX Runtime
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path)
out = sess.run(None, {'input': dummy_input.numpy()})
print(f"✅ ONNX Runtime Test: Output Sliders shape={out[0].shape}, Film shape={out[1].shape}")

## 💾 Bước 7: Tải Model Về Máy & Tích Hợp Vào Web IRO Studio

In [ ]:
try:
    from google.colab import files
    print("⬇️ Đang chuẩn bị tải file mô hình về máy tính của bạn...")
    files.download('iro_ai_enhancer.onnx')
    print("🎉 Sau khi tải xong, hãy copy file 'iro_ai_enhancer.onnx' vào thư mục 'web/models/' của dự án IRO Studio!")
except Exception as e:
    print("Bạn có thể tải file iro_ai_enhancer.onnx trực tiếp từ thanh menu Files bên trái Colab.")